# Ficheiro 2 - Morbilidade e Mortalidade Hospitalar

# Exploração dos dados (ler CSV e explorar)


In [4]:
import sqlite3
from pathlib import Path
import pandas as pd

df = pd.read_csv("../data/raw/morbilidade_mortalidade_hospit_fich2.csv",
                  sep=";", encoding="utf-8-sig")

# Renomear colunas para snake_case sem acentos/espaços
df.columns = ["periodo", "regiao", "instituicao", "codigo_diagnostico_principal", "descricao_diagnostico_principal", "faixa_etaria", "sexo", "internamentos", "dias_internamentos", "ambulatorio", "obitos"]

print("Informações do DataFrame")
print("")

print("Dimensão do DataFrame (linhas, colunas):")
print(df.shape)          # (linhas, colunas)
print("")

print("Primeiras 5 linhas: ")
display(df.head())         # primeiras 5 linhas
print("")

print("Estatísticas básicas: ")
display(df.describe().round(2))     # estatísticas básicas das colunas numéricas
print("")

print("Valores em falta por coluna:")
print(df.isna().sum())   # quantos valores em falta por coluna
print("")

print("Verificar se há linhas duplicadas: ")
print(df.duplicated().sum())
print("")

print("Informações: ")
display(df.info())    # colunas, tipos e não nulos
print("")     

print("Regiões:")
print(df["regiao"].unique()) # Ver que regiões são abordadas
print("")
print("")

print("Insituições:")
for inst in sorted(df["instituicao"].unique()):  # Ver que instituições são abordadas
    print(inst)


Informações do DataFrame

Dimensão do DataFrame (linhas, colunas):
(524406, 11)

Primeiras 5 linhas: 


,periodo,regiao,instituicao,codigo_diagnostico_principal,descricao_diagnostico_principal,faixa_etaria,sexo,internamentos,dias_internamentos,ambulatorio,obitos
0,2026-03,Região de Saúde do Norte,"Unidade Local de Saúde de Braga, E.P.E.",12,Doenças da pele e do tecido subcutâneo,[65-120[,M,2,24,0,0
1,2026-03,Região de Saúde do Algarve,"Unidade Local de Saúde do Algarve, E.P.E.",13,Doenças do aparelho osteomuscular e do tecido ...,[45-65[,M,2,3,1,0
2,2026-03,Região de Saúde do Alentejo,"Unidade Local de Saúde do Alto Alentejo, E.P.E.",13,Doenças do aparelho osteomuscular e do tecido ...,[45-65[,F,4,32,3,0
3,2026-03,Região de Saúde do Norte,"Unidade Local de Saúde do Alto Minho, E.P.E.",13,Doenças do aparelho osteomuscular e do tecido ...,[45-65[,F,1,2,8,0
4,2026-03,Região de Saúde do Norte,"Unidade Local de Saúde do Alto Minho, E.P.E.",13,Doenças do aparelho osteomuscular e do tecido ...,[65-120[,M,0,0,6,0



Estatísticas básicas: 


,codigo_diagnostico_principal,internamentos,dias_internamentos,ambulatorio,obitos
count,524406.00,524406.00,524406.00,524406.00,524406.00
mean,10.81,10.25,85.80,12.43,0.69
std,6.09,22.82,201.77,50.23,2.43
min,1.00,0.00,0.00,0.00,0.00
25%,6.00,1.00,3.00,0.00,0.00
50%,11.00,3.00,15.00,1.00,0.00
75%,15.00,8.00,71.00,5.00,0.00
max,22.00,470.00,14830.00,1267.00,120.00



Valores em falta por coluna:
periodo                            0
regiao                             0
instituicao                        0
codigo_diagnostico_principal       0
descricao_diagnostico_principal    0
faixa_etaria                       0
sexo                               0
internamentos                      0
dias_internamentos                 0
ambulatorio                        0
obitos                             0
dtype: int64

Verificar se há linhas duplicadas: 
0

Informações: 
<class 'pandas.DataFrame'>
RangeIndex: 524406 entries, 0 to 524405
Data columns (total 11 columns):
 #   Column                           Non-Null Count   Dtype
---  ------                           --------------   -----
 0   periodo                          524406 non-null  str  
 1   regiao                           524406 non-null  str  
 2   instituicao                      524406 non-null  str  
 3   codigo_diagnostico_principal     524406 non-null  int64
 4   descricao_diagnostico_pri

None


Regiões:
<StringArray>
[   'Região de Saúde do Norte',  'Região de Saúde do Algarve',
 'Região de Saúde do Alentejo',   'Região de Saúde do Centro',
         'Região de Saúde LVT']
Length: 5, dtype: str


Insituições:
Hospital de Cascais Dr. José de Almeida
Instituto Português Oncologia  F. Gentil - Coimbra, E.P.E.
Instituto Português Oncologia  F. Gentil - Lisboa, E.P.E.
Instituto Português Oncologia  F. Gentil - Porto, E.P.E.
Unidade Local de Saúde da Arrábida, E.P.E.
Unidade Local de Saúde da Cova da Beira, E.P.E.
Unidade Local de Saúde da Guarda, E.P.E.
Unidade Local de Saúde da Lezíria, E.P.E.
Unidade Local de Saúde da Póvoa de Varzim / Vila do Conde, E.P.E.
Unidade Local de Saúde da Região de Aveiro, E.P.E.
Unidade Local de Saúde da Região de Leiria, E.P.E.
Unidade Local de Saúde de Almada / Seixal, E.P.E.
Unidade Local de Saúde de Amadora / Sintra, E.P.E.
Unidade Local de Saúde de Barcelos / Esposende, E.P.E.
Unidade Local de Saúde de Braga, E.P.E.
Unidade Local de Saúde de Cas

# Limpeza e Transformação dos dados
## Preparação do ficheiro para análise:

In [5]:
# Eliminar a coluna "codigo_diagnostico_principal" 
df = df.drop(columns=["codigo_diagnostico_principal"])


#-------------------------------------------------------------------------------------------------
# Adicionar coluna descritiva com base na faixa etária
df["faixa_etaria_descricao"] = df["faixa_etaria"].replace({
    "[0-1[": "Lactentes",
    "[1-5[": "Primeira infância",
    "[5-15[": "Crianças",
    "[15-25[": "Adolescentes/Jovens",
    "[25-45[": "Adultos",
    "[45-65[": "Adultos de Meia-Idade",
    "[65-120[": "Idosos"
})
#-------------------------------------------------------------------------------------------------

# Verificar colunas atualizadas:
print("Colunas atualizadas (sem 'codigo_diagnostico_principal e com 'faixa_etaria_descricao'):")
print("")
print(df.columns)
print("")

#-------------------------------------------------------------------------------------------------
# Calcular demora média apenas onde doentes_saidos > 0
df["demora_media"] = (df["dias_internamentos"] / df["internamentos"]).where(df["internamentos"] > 0)
print("Demora média calculada (dias_internamentos / internamentos):")
print(df[["internamentos", "dias_internamentos", "demora_media"]].head())
print("")

#-------------------------------------------------------------------------------------------------

# Exportar ficheiro tratado
df.to_csv("../data/processed/morbilidade_mortalidade_tratado.csv", index=False)
print("Ficheiro exportado com sucesso!")

Colunas atualizadas (sem 'codigo_diagnostico_principal e com 'faixa_etaria_descricao'):

Index(['periodo', 'regiao', 'instituicao', 'descricao_diagnostico_principal',
       'faixa_etaria', 'sexo', 'internamentos', 'dias_internamentos',
       'ambulatorio', 'obitos', 'faixa_etaria_descricao'],
      dtype='str')

Demora média calculada (dias_internamentos / internamentos):
   internamentos  dias_internamentos  demora_media
0              2                  24          12.0
1              2                   3           1.5
2              4                  32           8.0
3              1                   2           2.0
4              0                   0           NaN

Ficheiro exportado com sucesso!
